# Feature Engineering

This notebook prepares a first machine-learning-ready feature dataset for the AI-driven Loan Analytics Platform.

The objective is to create a professional baseline feature set for default-risk modeling while avoiding target leakage.

Scope:

- define the prediction target;
- separate safe, risky, and excluded variables;
- create interpretable borrower, loan, date, and affordability features;
- preserve missing-value meaning with missingness indicators;
- save a modeling dataset for the next notebook;
- save a feature dictionary for auditability.

This notebook does not train a machine learning model.

## 1. Imports and configuration

We load the cleaned dataset from `data/processed` and define output paths for the modeling dataset and feature dictionary.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'data_cleaned.csv'
OUTPUT_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'modeling_dataset_baseline.csv'
FEATURE_DICTIONARY_PATH = PROJECT_ROOT / 'reports' / 'feature_dictionary.csv'
FEATURE_CONFIG_PATH = PROJECT_ROOT / 'reports' / 'feature_config_baseline.json'

TARGET_COLUMN = 'has_default_within_12_months'
RANDOM_STATE = 42

print(f'Input path: {INPUT_PATH}')
print(f'Output modeling dataset: {OUTPUT_DATA_PATH}')
print(f'Feature dictionary: {FEATURE_DICTIONARY_PATH}')

Input path: C:\Users\aitmo\financial-loan-analytics\data\processed\data_cleaned.csv
Output modeling dataset: C:\Users\aitmo\financial-loan-analytics\data\processed\modeling_dataset_baseline.csv
Feature dictionary: C:\Users\aitmo\financial-loan-analytics\reports\feature_dictionary.csv


## 1.1 Helper functions

Helper functions keep repeated checks compact and readable.

In [2]:
def missing_table(dataframe):
    """Return a compact missing-value summary for a dataframe."""
    table = pd.DataFrame({
        'column': dataframe.columns,
        'missing_values': dataframe.isna().sum().values,
        'missing_percentage': dataframe.isna().mean().values,
        'dtype': dataframe.dtypes.astype(str).values,
        'unique_values': dataframe.nunique(dropna=True).values,
    })
    return table.sort_values('missing_percentage', ascending=False)


print('Helper functions loaded.')

Helper functions loaded.


## 2. Load cleaned dataset

The cleaned dataset is used as input. The raw dataset remains untouched.

In [3]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
display(df.head())

Rows: 737,889
Columns: 31


,loan_id,country,loan_issued_at,early_repaid_at,is_early_repaid_within_14_days,issued_amount,loan_status,loan_last_recorded_action_date_local,initial_interest_rate,nr_of_payments,principal_balance,principal_debt,principal_paid_total,interest_paid_total,extra_interest_paid_total,late_fee_paid_total,maintenance_fee_paid_total,is_default,next_payment_nr,next_payment_date_local,debt_occured_date_local,days_past_due_principal,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,projected_npv_return,customer_risk_rating
0,0B3B642A-F923-49AD-84ED-B2A700BF1D0A,Finland,2025-03-21 11:45:56.040,2025-04-04 10:50:01.137,1.0000,310.0000,Repaid,2025-04-04,0.1780,1,0.0000,0.0000,310.0000,1.9900,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,Cancelled,311.9900,12,"2,068.6800",0.0000,0.0189,F
1,4FD1927C-0E7F-4BF1-AED0-B268011E3E85,Finland,2025-01-17 17:27:25.447,NaN,NaN,414.0000,Active,NaN,0.1780,120,301.4900,0.0000,112.5100,70.8300,0.0000,0.0000,3.2700,0.0000,11.0000,2026-05-08,NaN,NaN,NaN,15.0000,Active,186.6100,18,"1,469.5000",0.0000,0.1242,B
2,7362E260-54DC-4798-8848-B18C00A59FCB,Finland,2024-06-11 10:04:52.427,2025-01-02 12:07:32.807,0.0000,518.0000,Repaid,2025-01-02,0.1877,8,0.0000,0.0000,518.0000,51.6000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,6.0000,Paid Up,569.6000,60,"4,513.2000",0.0000,0.1200,B
3,8C7E5FB6-A673-4148-BCCF-B27500B0368E,Finland,2025-01-30 10:52:12.657,NaN,NaN,"3,731.0000",Active,NaN,0.1780,75,"3,305.8700",0.0000,425.1300,724.9300,0.0000,0.0000,22.6300,0.0000,14.0000,2026-05-04,NaN,NaN,NaN,15.0000,Active,"1,172.6900",60,"1,301.0000",0.0000,0.1014,C
4,BE865034-6041-4855-AB42-B196014FA0E8,Netherlands,2024-06-21 20:48:23.787,NaN,NaN,"3,432.0000",Active,NaN,0.1056,38,"2,371.8200",0.0000,"1,060.1800",549.6600,0.2900,0.0000,60.6500,0.0000,1.0000,2026-05-01,NaN,NaN,NaN,22.0000,Active,"1,670.7800",60,"5,194.7000",0.0000,0.1094,A


## 3. Modeling objective and target policy

For the baseline supervised ML dataset, the target is `has_default_within_12_months`.

Rows with missing target labels are excluded only from the modeling dataset. They remain in the cleaned dataset and can still be useful for unsupervised analysis, monitoring, or dashboarding.

In [4]:
if TARGET_COLUMN not in df.columns:
    raise KeyError(f'Target column not found: {TARGET_COLUMN}')

target_distribution = (
    df[TARGET_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COLUMN)
    .reset_index(name='count')
)
target_distribution['percentage'] = target_distribution['count'] / len(df)

display(target_distribution)

model_df = df[df[TARGET_COLUMN].notna()].copy()
model_df[TARGET_COLUMN] = model_df[TARGET_COLUMN].astype(int)

print(f'Rows before target filtering: {len(df):,}')
print(f'Rows after target filtering: {len(model_df):,}')
print(f'Removed rows with missing target: {len(df) - len(model_df):,}')

,has_default_within_12_months,count,percentage
0,0.0000,480438,0.6511
1,NaN,174379,0.2363
2,1.0000,83072,0.1126


Rows before target filtering: 737,889
Rows after target filtering: 563,510
Removed rows with missing target: 174,379


## 4. Leakage and feature availability policy

A baseline default-risk model should use information that is likely available at or near origination.

We intentionally exclude fields that directly describe default, delinquency, repayment behavior, or future events. Those fields can be useful for portfolio monitoring, but they are risky for default prediction because they may leak the answer.

In [5]:
target_columns = [
    'has_default_within_12_months',
]

high_leakage_columns = [
    'is_default',
    'loan_status',
    'loan_status_risk',
    'months_in_default',
    'days_past_due_principal',
    'debt_occured_date_local',
    'principal_debt',
]

post_origination_columns = [
    'early_repaid_at',
    'is_early_repaid_within_14_days',
    'loan_last_recorded_action_date_local',
    'next_payment_nr',
    'next_payment_date_local',
    'principal_balance',
    'principal_paid_total',
    'interest_paid_total',
    'extra_interest_paid_total',
    'late_fee_paid_total',
    'maintenance_fee_paid_total',
    'repaid_amount_total',
    'months_on_book',
]

business_review_columns = [
    'customer_risk_rating',
    'projected_npv_return',
]

candidate_origination_columns = [
    'loan_id',
    'country',
    'loan_issued_at',
    'issued_amount',
    'initial_interest_rate',
    'nr_of_payments',
    'initial_loan_duration',
    'combined_income',
]

availability_policy = pd.DataFrame([
    {'column': column, 'policy': 'target', 'include_baseline': False}
    for column in target_columns if column in df.columns
] + [
    {'column': column, 'policy': 'exclude_high_leakage', 'include_baseline': False}
    for column in high_leakage_columns if column in df.columns
] + [
    {'column': column, 'policy': 'exclude_post_origination', 'include_baseline': False}
    for column in post_origination_columns if column in df.columns
] + [
    {'column': column, 'policy': 'business_review_optional', 'include_baseline': False}
    for column in business_review_columns if column in df.columns
] + [
    {'column': column, 'policy': 'candidate_origination_safe', 'include_baseline': True}
    for column in candidate_origination_columns if column in df.columns
])

display(availability_policy)

,column,policy,include_baseline
0,has_default_within_12_months,target,False
1,is_default,exclude_high_leakage,False
2,loan_status,exclude_high_leakage,False
3,loan_status_risk,exclude_high_leakage,False
4,months_in_default,exclude_high_leakage,False
5,days_past_due_principal,exclude_high_leakage,False
6,debt_occured_date_local,exclude_high_leakage,False
7,principal_debt,exclude_high_leakage,False
8,early_repaid_at,exclude_post_origination,False
9,is_early_repaid_within_14_days,exclude_post_origination,False


## 5. Create baseline feature frame

The baseline feature frame starts from conservative origination-safe variables. We keep `loan_id` only as an identifier, not as a predictive feature.

In [6]:
feature_df = pd.DataFrame(index=model_df.index)

if 'loan_id' in model_df.columns:
    feature_df['loan_id'] = model_df['loan_id']

# Target
feature_df[TARGET_COLUMN] = model_df[TARGET_COLUMN]

# Core origination-safe raw variables
for column in [
    'country',
    'issued_amount',
    'initial_interest_rate',
    'nr_of_payments',
    'initial_loan_duration',
    'combined_income',
]:
    if column in model_df.columns:
        feature_df[column] = model_df[column]

print(f'Baseline feature frame shape: {feature_df.shape}')
display(feature_df.head())

Baseline feature frame shape: (563510, 8)


,loan_id,has_default_within_12_months,country,issued_amount,initial_interest_rate,nr_of_payments,initial_loan_duration,combined_income
0,0B3B642A-F923-49AD-84ED-B2A700BF1D0A,0,Finland,310.0000,0.1780,1,12,"2,068.6800"
1,4FD1927C-0E7F-4BF1-AED0-B268011E3E85,0,Finland,414.0000,0.1780,120,18,"1,469.5000"
2,7362E260-54DC-4798-8848-B18C00A59FCB,0,Finland,518.0000,0.1877,8,60,"4,513.2000"
3,8C7E5FB6-A673-4148-BCCF-B27500B0368E,0,Finland,"3,731.0000",0.1780,75,60,"1,301.0000"
4,BE865034-6041-4855-AB42-B196014FA0E8,0,Netherlands,"3,432.0000",0.1056,38,60,"5,194.7000"


## 6. Date-based origination features

Date features are derived from `loan_issued_at`, which should be known at origination. These features can capture seasonality, time periods, and cohort effects.

In [7]:
if 'loan_issued_at' in model_df.columns:
    issue_date = pd.to_datetime(model_df['loan_issued_at'], errors='coerce')

    feature_df['issue_year'] = issue_date.dt.year
    feature_df['issue_month'] = issue_date.dt.month
    feature_df['issue_quarter'] = issue_date.dt.quarter
    feature_df['issue_day_of_week'] = issue_date.dt.dayofweek
    feature_df['issue_month_sin'] = np.sin(2 * np.pi * feature_df['issue_month'] / 12)
    feature_df['issue_month_cos'] = np.cos(2 * np.pi * feature_df['issue_month'] / 12)
    feature_df['loan_issued_at_missing'] = issue_date.isna().astype(int)
else:
    print('loan_issued_at was not found; no date features created.')

display(feature_df.filter(like='issue').head())

,issued_amount,issue_year,issue_month,issue_quarter,issue_day_of_week,issue_month_sin,issue_month_cos,loan_issued_at_missing
0,310.0000,2025,3,1,4,1.0000,0.0000,0
1,414.0000,2025,1,1,4,0.5000,0.8660,0
2,518.0000,2024,6,2,1,0.0000,-1.0000,0
3,"3,731.0000",2025,1,1,3,0.5000,0.8660,0
4,"3,432.0000",2024,6,2,4,0.0000,-1.0000,0


## 7. Loan size, affordability, and pricing features

These features describe loan amount, pricing, duration, and repayment burden proxies.

They are designed to support default-risk modeling, but the final usefulness must be tested in the modeling phase.

In [8]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


if 'issued_amount' in feature_df.columns:
    feature_df['log_issued_amount'] = np.log1p(feature_df['issued_amount'].clip(lower=0))

if 'combined_income' in feature_df.columns:
    feature_df['combined_income_missing'] = feature_df['combined_income'].isna().astype(int)
    feature_df['log_combined_income'] = np.log1p(feature_df['combined_income'].clip(lower=0))

if {'issued_amount', 'combined_income'}.issubset(feature_df.columns):
    feature_df['loan_to_income_ratio'] = safe_divide(
        feature_df['issued_amount'],
        feature_df['combined_income'],
    )

if {'issued_amount', 'nr_of_payments'}.issubset(feature_df.columns):
    feature_df['installment_amount_proxy'] = safe_divide(
        feature_df['issued_amount'],
        feature_df['nr_of_payments'],
    )

if {'installment_amount_proxy', 'combined_income'}.issubset(feature_df.columns):
    feature_df['installment_to_income_proxy'] = safe_divide(
        feature_df['installment_amount_proxy'],
        feature_df['combined_income'],
    )

if {'issued_amount', 'initial_interest_rate'}.issubset(feature_df.columns):
    feature_df['interest_cost_proxy'] = (
        feature_df['issued_amount'] * feature_df['initial_interest_rate']
    )

if {'issued_amount', 'initial_loan_duration'}.issubset(feature_df.columns):
    feature_df['amount_per_duration_unit'] = safe_divide(
        feature_df['issued_amount'],
        feature_df['initial_loan_duration'],
    )

engineered_numeric_features = [
    column for column in [
        'log_issued_amount',
        'combined_income_missing',
        'log_combined_income',
        'loan_to_income_ratio',
        'installment_amount_proxy',
        'installment_to_income_proxy',
        'interest_cost_proxy',
        'amount_per_duration_unit',
    ]
    if column in feature_df.columns
]

display(feature_df[engineered_numeric_features].head())

,log_issued_amount,combined_income_missing,log_combined_income,loan_to_income_ratio,installment_amount_proxy,installment_to_income_proxy,interest_cost_proxy,amount_per_duration_unit
0,5.7398,0,7.6351,0.1499,310.0000,0.1499,55.1800,25.8333
1,6.0283,0,7.2934,0.2817,3.4500,0.0023,73.6920,23.0000
2,6.2519,0,8.4150,0.1148,64.7500,0.0143,97.2286,8.6333
3,8.2247,0,7.1717,2.8678,49.7467,0.0382,664.1180,62.1833
4,8.1412,0,8.5556,0.6607,90.3158,0.0174,362.4192,57.2000


## 8. Categorical feature handling

For the baseline dataset, categorical variables are kept as readable categories. Encoding should happen inside the modeling pipeline, not here.

Missing categorical values are represented with a clear category label so models and dashboards can distinguish them from known categories.

In [9]:
categorical_feature_columns = [
    column for column in [
        'country',
    ]
    if column in feature_df.columns
]

for column in categorical_feature_columns:
    feature_df[column] = feature_df[column].fillna('Missing / not recorded').astype(str)

categorical_summary = pd.DataFrame([
    {
        'column': column,
        'unique_values': feature_df[column].nunique(dropna=False),
        'top_category': feature_df[column].value_counts(dropna=False).index[0],
        'top_category_share': feature_df[column].value_counts(dropna=False).iloc[0] / len(feature_df),
    }
    for column in categorical_feature_columns
])

categorical_summary

,column,unique_values,top_category,top_category_share
0,country,8,Finland,0.5065


## 9. Missing-value indicators and remaining missingness

We create explicit missingness indicators for important numeric fields. Numeric missing values themselves are kept as missing so the modeling pipeline can impute them after train/test split, avoiding leakage.

In [10]:
numeric_missing_indicator_columns = []
for column in [
    'issued_amount',
    'initial_interest_rate',
    'nr_of_payments',
    'initial_loan_duration',
    'combined_income',
    'loan_to_income_ratio',
    'installment_to_income_proxy',
]:
    if column in feature_df.columns:
        indicator_column = f'{column}_missing'
        if indicator_column not in feature_df.columns:
            feature_df[indicator_column] = feature_df[column].isna().astype(int)
        numeric_missing_indicator_columns.append(indicator_column)

remaining_missing = missing_table(feature_df)
remaining_missing_nonzero = remaining_missing[
    remaining_missing['missing_values'] > 0
]

display(remaining_missing_nonzero)
print(f'Missingness indicator columns created: {len(numeric_missing_indicator_columns)}')

,column,missing_values,missing_percentage,dtype,unique_values
20,installment_to_income_proxy,273426,0.4852,float64,254746
18,loan_to_income_ratio,273426,0.4852,float64,240958
17,log_combined_income,273154,0.4847,float64,148718
7,combined_income,273154,0.4847,float64,148718


Missingness indicator columns created: 7


## 10. Baseline feature list

This section defines the columns that will be used by the next modeling notebook.

`loan_id` is retained as an identifier but should not be used as a model input.

In [11]:
identifier_columns = [column for column in ['loan_id'] if column in feature_df.columns]
target_columns = [TARGET_COLUMN]

model_input_features = [
    column for column in feature_df.columns
    if column not in identifier_columns + target_columns
]

feature_config = {
    'target_column': TARGET_COLUMN,
    'identifier_columns': identifier_columns,
    'model_input_features': model_input_features,
    'categorical_features': [
        column for column in categorical_feature_columns if column in model_input_features
    ],
    'numeric_features': [
        column for column in model_input_features
        if column not in categorical_feature_columns
    ],
    'excluded_high_leakage_columns': [
        column for column in high_leakage_columns if column in df.columns
    ],
    'excluded_post_origination_columns': [
        column for column in post_origination_columns if column in df.columns
    ],
    'business_review_columns_not_in_baseline': [
        column for column in business_review_columns if column in df.columns
    ],
}

print(f'Model input features: {len(model_input_features)}')
print(f'Numeric features: {len(feature_config["numeric_features"])}')
print(f'Categorical features: {len(feature_config["categorical_features"])}')

display(pd.DataFrame({'model_input_feature': model_input_features}))

Model input features: 27
Numeric features: 26
Categorical features: 1


,model_input_feature
0,country
1,issued_amount
2,initial_interest_rate
3,nr_of_payments
4,initial_loan_duration
5,combined_income
6,issue_year
7,issue_month
8,issue_quarter
9,issue_day_of_week


## 11. Feature dictionary

The feature dictionary documents each output column, its role, source, and modeling notes. This is important for a professional ML workflow and future auditability.

In [12]:
feature_dictionary_rows = []

for column in feature_df.columns:
    if column in identifier_columns:
        role = 'identifier'
        source = 'original'
        modeling_note = 'Keep for traceability; do not use as model input.'
    elif column == TARGET_COLUMN:
        role = 'target'
        source = 'original'
        modeling_note = 'Prediction target.'
    elif column in df.columns:
        role = 'feature'
        source = 'original_baseline_candidate'
        modeling_note = 'Candidate model input; confirm availability at prediction time.'
    else:
        role = 'feature'
        source = 'engineered'
        modeling_note = 'Created in feature engineering notebook.'

    if column.endswith('_missing'):
        modeling_note = 'Missingness indicator; useful when missing values carry business meaning.'

    feature_dictionary_rows.append({
        'column': column,
        'role': role,
        'source': source,
        'dtype': str(feature_df[column].dtype),
        'missing_values': int(feature_df[column].isna().sum()),
        'missing_percentage': feature_df[column].isna().mean(),
        'unique_values': int(feature_df[column].nunique(dropna=True)),
        'included_as_model_input': column in model_input_features,
        'modeling_note': modeling_note,
    })

feature_dictionary = pd.DataFrame(feature_dictionary_rows)
display(feature_dictionary)

,column,role,source,dtype,missing_values,missing_percentage,unique_values,included_as_model_input,modeling_note
0,loan_id,identifier,original,object,0,0.0000,563510,False,Keep for traceability; do not use as model input.
1,has_default_within_12_months,target,original,int64,0,0.0000,2,False,Prediction target.
2,country,feature,original_baseline_candidate,object,0,0.0000,8,True,Candidate model input; confirm availability at...
3,issued_amount,feature,original_baseline_candidate,float64,0,0.0000,7658,True,Candidate model input; confirm availability at...
4,initial_interest_rate,feature,original_baseline_candidate,float64,0,0.0000,6258,True,Candidate model input; confirm availability at...
5,nr_of_payments,feature,original_baseline_candidate,int64,0,0.0000,142,True,Candidate model input; confirm availability at...
6,initial_loan_duration,feature,original_baseline_candidate,int64,0,0.0000,77,True,Candidate model input; confirm availability at...
7,combined_income,feature,original_baseline_candidate,float64,273154,0.4847,148718,True,Candidate model input; confirm availability at...
8,issue_year,feature,engineered,int32,0,0.0000,17,True,Created in feature engineering notebook.
9,issue_month,feature,engineered,int32,0,0.0000,12,True,Created in feature engineering notebook.


## 12. Quality checks before saving

Before saving, we run basic checks to make sure the modeling dataset has a target, at least one feature, no duplicate column names, and no missing target values.

In [13]:
quality_checks = {
    'rows': len(feature_df),
    'columns': feature_df.shape[1],
    'model_input_features': len(model_input_features),
    'missing_target_values': int(feature_df[TARGET_COLUMN].isna().sum()),
    'duplicate_column_names': int(pd.Index(feature_df.columns).duplicated().sum()),
    'duplicate_rows': int(feature_df.duplicated().sum()),
}

quality_checks_df = pd.DataFrame(
    quality_checks.items(),
    columns=['check', 'value'],
)

display(quality_checks_df)

if quality_checks['missing_target_values'] != 0:
    raise ValueError('Modeling dataset contains missing target values.')
if quality_checks['model_input_features'] == 0:
    raise ValueError('No model input features were created.')
if quality_checks['duplicate_column_names'] != 0:
    raise ValueError('Duplicate column names found in feature dataset.')

,check,value
0,rows,563510
1,columns,29
2,model_input_features,27
3,missing_target_values,0
4,duplicate_column_names,0
5,duplicate_rows,0


## 13. Save feature engineering outputs

We save three outputs:

- `modeling_dataset_baseline.csv`: conservative baseline dataset for ML;
- `feature_dictionary.csv`: feature documentation;
- `feature_config_baseline.json`: feature lists for the modeling notebook.

In [14]:
OUTPUT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATURE_DICTIONARY_PATH.parent.mkdir(parents=True, exist_ok=True)

feature_df.to_csv(OUTPUT_DATA_PATH, index=False)
feature_dictionary.to_csv(FEATURE_DICTIONARY_PATH, index=False)

with open(FEATURE_CONFIG_PATH, 'w', encoding='utf-8') as file:
    json.dump(feature_config, file, indent=4)

print(f'Modeling dataset saved to: {OUTPUT_DATA_PATH}')
print(f'Feature dictionary saved to: {FEATURE_DICTIONARY_PATH}')
print(f'Feature config saved to: {FEATURE_CONFIG_PATH}')

Modeling dataset saved to: C:\Users\aitmo\financial-loan-analytics\data\processed\modeling_dataset_baseline.csv
Feature dictionary saved to: C:\Users\aitmo\financial-loan-analytics\reports\feature_dictionary.csv
Feature config saved to: C:\Users\aitmo\financial-loan-analytics\reports\feature_config_baseline.json


## 14. Feature engineering summary and next steps

The baseline feature engineering phase is complete.

Next recommended step: create a modeling notebook that loads `modeling_dataset_baseline.csv`, performs train/test split, builds preprocessing pipelines, handles imputation inside the pipeline, trains baseline models, and evaluates performance with default-risk metrics.

In [15]:
summary_lines = [
    f'- Modeling rows: **{len(feature_df):,}**.',
    f'- Total columns saved: **{feature_df.shape[1]}**.',
    f'- Model input features: **{len(model_input_features)}**.',
    f'- Target column: **`{TARGET_COLUMN}`**.',
    f'- Categorical features: **{len(feature_config["categorical_features"])}**.',
    f'- Numeric features: **{len(feature_config["numeric_features"])}**.',
    '- Numeric missing values were not imputed here; imputation should happen inside the modeling pipeline.',
    '- High-leakage and post-origination variables were excluded from the conservative baseline dataset.',
]

display(Markdown(chr(10).join(summary_lines)))

- Modeling rows: **563,510**.
- Total columns saved: **29**.
- Model input features: **27**.
- Target column: **`has_default_within_12_months`**.
- Categorical features: **1**.
- Numeric features: **26**.
- Numeric missing values were not imputed here; imputation should happen inside the modeling pipeline.
- High-leakage and post-origination variables were excluded from the conservative baseline dataset.